# 08 · Distribution Shift Detection

## What this notebook covers
Models are trained on a snapshot of the world. When the world changes — seasonality, market shifts, population changes, upstream data pipeline changes — the model's input distribution changes and performance degrades silently. This notebook implements the four primary shift detection methods used in production ML monitoring.

## The shift taxonomy
| Type | What drifts | Typical cause |
|------|------------|---------------|
| **Covariate shift** | P(X) changes, P(y|X) stable | New user segment, seasonality |
| **Label shift** | P(y) changes | Market event, class prevalence change |
| **Concept drift** | P(y|X) changes | World changes, adversarial users |
| **Data pipeline drift** | Feature schema/distribution changes | ETL bug, upstream schema change |

In practice you detect *feature distribution changes* (covariate shift proxy) because labels are often delayed or unavailable in production.

## Detection methods implemented here
- **KS test** — per-feature, non-parametric, detects any distributional change
- **JS divergence** — symmetric, bounded [0, log2], captures full distribution shape
- **MMD (Maximum Mean Discrepancy)** — kernel-based, detects higher-order differences
- **PSI (Population Stability Index)** — industry standard in credit/insurance; detects category shift

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics.pairwise import rbf_kernel
from collections import deque
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# Reference distribution (training time)
n_ref = 1000
X_ref = np.column_stack([
    np.random.normal(0, 1, n_ref),
    np.random.normal(5, 2, n_ref),
    np.random.exponential(2, n_ref),
    np.random.uniform(0, 10, n_ref),
])
feature_names = ["age_norm", "income_norm", "tenure", "score"]

# Production distribution: features 0 and 2 have drifted
X_prod = np.column_stack([
    np.random.normal(0.8, 1.2, n_ref),  # drifted
    np.random.normal(5, 2, n_ref),       # stable
    np.random.exponential(3, n_ref),     # drifted
    np.random.uniform(0, 10, n_ref),     # stable
])

In [ ]:
def ks_test_features(X_ref, X_prod, feature_names, alpha=0.05):
    """Per-feature KS test. Returns DataFrame with stat, p-value, drift flag."""
    results = []
    for i, name in enumerate(feature_names):
        stat, p = stats.ks_2samp(X_ref[:, i], X_prod[:, i])
        results.append({"feature": name, "ks_stat": round(stat, 4), "p_value": round(p, 4), "drift": p < alpha})
    return pd.DataFrame(results)

ks_results = ks_test_features(X_ref, X_prod, feature_names)
print(ks_results.to_string(index=False))

In [ ]:
def js_divergence(p_sample, q_sample, bins=50):
    """JS divergence between two samples via histogram approximation."""
    lo = min(p_sample.min(), q_sample.min())
    hi = max(p_sample.max(), q_sample.max())
    p_hist, _ = np.histogram(p_sample, bins=bins, range=(lo, hi), density=True)
    q_hist, _ = np.histogram(q_sample, bins=bins, range=(lo, hi), density=True)
    p_hist = p_hist + 1e-10
    q_hist = q_hist + 1e-10
    p_hist /= p_hist.sum()
    q_hist /= q_hist.sum()
    m = 0.5 * (p_hist + q_hist)
    kl_pm = (p_hist * np.log(p_hist / m)).sum()
    kl_qm = (q_hist * np.log(q_hist / m)).sum()
    return 0.5 * (kl_pm + kl_qm)

def mmd_rbf(X, Y, gamma=None):
    """Maximum Mean Discrepancy with RBF kernel."""
    if gamma is None:
        gamma = 1.0 / X.shape[1]
    XX = rbf_kernel(X, X, gamma)
    YY = rbf_kernel(Y, Y, gamma)
    XY = rbf_kernel(X, Y, gamma)
    return XX.mean() + YY.mean() - 2 * XY.mean()

def psi(expected, actual, bins=10):
    """Population Stability Index. PSI > 0.2 = significant shift."""
    lo = min(expected.min(), actual.min())
    hi = max(expected.max(), actual.max())
    exp_hist, edges = np.histogram(expected, bins=bins, range=(lo, hi))
    act_hist, _     = np.histogram(actual,   bins=bins, range=(lo, hi))
    exp_pct = (exp_hist + 1) / (exp_hist + 1).sum()
    act_pct = (act_hist + 1) / (act_hist + 1).sum()
    return ((act_pct - exp_pct) * np.log(act_pct / exp_pct)).sum()

print("JS divergence per feature:")
for i, name in enumerate(feature_names):
    jsd = js_divergence(X_ref[:, i], X_prod[:, i])
    psi_val = psi(X_ref[:, i], X_prod[:, i])
    print(f"  {name:<15}: JS={jsd:.4f}  PSI={psi_val:.4f}  {'⚠ DRIFT' if psi_val > 0.2 else 'OK'}")

print(f"\nMMD (all features): {mmd_rbf(X_ref[:200], X_prod[:200]):.6f}")

In [ ]:
class DriftMonitor:
    """Sliding window drift monitor for production use."""
    def __init__(self, reference, feature_names, window_size=500, ks_alpha=0.05):
        self.reference     = reference
        self.feature_names = feature_names
        self.window_size   = window_size
        self.ks_alpha      = ks_alpha
        self.buffer        = deque(maxlen=window_size)
        self.alerts        = []

    def update(self, batch, batch_id=0):
        """Add a batch of rows and check for drift."""
        for row in batch:
            self.buffer.append(row)
        if len(self.buffer) < self.window_size:
            return None
        window_arr = np.array(self.buffer)
        results = ks_test_features(self.reference, window_arr, self.feature_names, self.ks_alpha)
        drifted = results[results["drift"]]["feature"].tolist()
        entry = {"batch_id": batch_id, "window_n": len(self.buffer), "drifted_features": drifted}
        if drifted:
            self.alerts.append(entry)
        return entry

monitor = DriftMonitor(X_ref, feature_names, window_size=500)
for i, batch in enumerate(np.array_split(X_prod, 4)):
    result = monitor.update(batch, batch_id=i)
    if result:
        flag = "⚠ ALERT" if result["drifted_features"] else "OK"
        print(f"Batch {i}: {flag}  drifted={result['drifted_features']}")